# Partie III — RNN, LSTM, GRU et Seq2Seq
## Classification et résumé de titres d'actualités climatiques — Dataset *Climate News Articles* (Kaggle)

**EMSI Casablanca — Module Deep Learning — Année universitaire 2025-2026**

**Réalisé par : Imane**

---

### Objectif
Ce notebook modélise des séquences textuelles réelles (titres et descriptions d'articles
d'actualité liés au climat et à l'environnement). Deux tâches sont traitées :

1. **Classification de séquences** : prédire la catégorie thématique d'un article à partir de
   son titre, en comparant RNN simple, LSTM et GRU.
2. **Seq2Seq (résumé automatique)** : générer une version courte (résumé) d'une description
   d'article à partir d'un encodeur-décodeur LSTM avec teacher forcing, évalué avec BLEU et
   deux stratégies de décodage (glouton et beam search).

Le notebook couvre :
1. Modèle de langage probabiliste, factorisation par la règle de chaîne, perplexité
2. RNN, LSTM, GRU — implémentation et comparaison (stabilité, mémoire, coût)
3. BPTT et effet du gradient clipping
4. Préparation des données (tokenisation, vocabulaire, padding, masquage, mini-lots)
5. Architecture Seq2Seq encodeur-décodeur avec teacher forcing
6. Décodage glouton vs beam search, évaluation BLEU
7. Analyse critique et question de synthèse


## 1. Configuration et reproductibilité

In [ ]:
!pip install -q torch scikit-learn matplotlib seaborn pandas nltk kagglehub

In [ ]:
import os
import random
import re
import math
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smoothie = SmoothingFunction().method4

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {DEVICE}")


## 2. Modèle de langage probabiliste

### 2.1 Factorisation par la règle de chaîne
Un modèle de langage attribue une probabilité à une séquence de tokens
$(x_1, x_2, \dots, x_T)$ via la règle de chaîne :

$$P(x_1, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1})$$

Un RNN/LSTM/GRU approxime chaque terme $P(x_t \mid x_{<t})$ à l'aide d'un état caché
$h_{t-1}$ qui résume l'historique : $P(x_t \mid x_{<t}) \approx P(x_t \mid h_{t-1})$.

### 2.2 Perplexité
La perplexité est l'exponentielle de la loss moyenne (cross-entropy) :
$$\text{PPL} = \exp\left(\frac{1}{T}\sum_{t=1}^{T} -\log P(x_t \mid x_{<t})\right)$$

Elle s'interprète comme le nombre moyen de choix "équiprobables" parmi lesquels le modèle
hésite à chaque pas de temps. Une perplexité de 1 correspond à une prédiction parfaite ; plus
la perplexité est faible, meilleur est le modèle.


## 3. Chargement du dataset

**Dataset : Climate Change News Articles** (Kaggle, `amritpal333/climate-news-articles`)

Articles d'actualité liés au climat et à l'environnement, avec titre, description et source.
Pour la classification, des catégories thématiques (Climate Policy, Renewable Energy,
Wildlife & Ecosystems, Extreme Weather, Pollution) sont dérivées à partir de mots-clés
présents dans le texte. Pour le Seq2Seq, la tâche consiste à générer un résumé court
(le titre) à partir de la description complète.

Ordre de chargement :
1. fichier local `data/climate_news.csv` (fourni dans le dépôt, déjà étiqueté
   par catégorie) ;
2. à défaut, téléchargement Kaggle via `kagglehub` (étiquetage par mots-clés appliqué
   ensuite) ;
3. à défaut, génération synthétique de titres/descriptions d'actualités climatiques
   (combinatoire de modèles de phrases, SEED=42).


In [ ]:
CATEGORIES = ["Climate Policy", "Renewable Energy", "Wildlife & Ecosystems",
              "Extreme Weather", "Pollution"]

KEYWORDS = {
    "Climate Policy": ["agreement", "policy", "summit", "regulation", "treaty", "government",
                        "law", "negotiations", "cop", "pledge"],
    "Renewable Energy": ["solar", "wind", "renewable", "battery", "energy", "turbine",
                          "hydrogen", "grid", "panel"],
    "Wildlife & Ecosystems": ["species", "wildlife", "forest", "biodiversity", "habitat",
                               "coral", "ocean", "ecosystem", "extinction"],
    "Extreme Weather": ["storm", "hurricane", "flood", "drought", "heatwave", "wildfire",
                         "temperature", "rainfall", "cyclone"],
    "Pollution": ["pollution", "emissions", "plastic", "waste", "air quality", "carbon",
                   "toxic", "contamination"],
}

LOCAL_CSV = "../data/climate_news.csv"

if os.path.exists(LOCAL_CSV):
    df = pd.read_csv(LOCAL_CSV)
    print(f"Chargement depuis le fichier local : {LOCAL_CSV}")
else:
    try:
        import kagglehub
        path = kagglehub.dataset_download("amritpal333/climate-news-articles")
        print("Dataset téléchargé dans :", path)
        csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
        df = pd.read_csv(os.path.join(path, csv_files[0]))
        df = df.rename(columns={c: c.lower() for c in df.columns})
        text_col = "description" if "description" in df.columns else df.columns[1]
        title_col = "title" if "title" in df.columns else df.columns[0]
        df = df[[title_col, text_col]].dropna().rename(
            columns={title_col: "title", text_col: "description"})
        print("Chargement Kaggle réussi.")

    except Exception as e:
        print(f"Téléchargement Kaggle indisponible ({e}). "
              f"Génération d'un corpus synthétique d'actualités climatiques (SEED=42) en secours.")

        rng = random.Random(SEED)

        # Templates plus variés et ambigus — mots clés partagés entre catégories
        templates = [
            "{subject} warns about rising {issue} levels affecting {region}",
            "new study reveals {issue} threatens {target} in {region}",
            "{region} officials debate {issue} amid growing {concern}",
            "scientists link {issue} to changes in {target} across {region}",
            "{subject} calls for action on {issue} following {event} in {region}",
            "report shows {issue} impacting {target} and local communities",
            "{region} struggles with {issue} as {concern} rises",
            "activists demand response to {issue} affecting {target}",
            "{subject} releases data on {issue} linked to {concern}",
            "global summit addresses {issue} and its effects on {target}",
        ]

        # Topics ambigus — certains mots apparaissent dans plusieurs catégories
        topics_by_cat = {
            "Climate Policy": {
                "issue":   ["carbon targets", "emission regulations", "climate commitments",
                             "net-zero pledges", "environmental legislation", "policy frameworks",
                             "international agreements", "carbon trading"],
                "target":  ["developing nations", "industries", "city governments",
                             "coastal communities", "rural populations"],
                "concern": ["political resistance", "economic impact", "lack of consensus",
                             "funding gaps", "implementation delays"],
                "event":   ["the COP summit", "a senate hearing", "a UN conference",
                             "bilateral talks", "a ministerial meeting"],
            },
            "Renewable Energy": {
                "issue":   ["solar capacity", "wind farm expansion", "battery storage",
                             "grid modernization", "clean energy transition",
                             "hydrogen production", "offshore wind projects"],
                "target":  ["energy grids", "power infrastructure", "rural electrification",
                             "industrial sectors", "transport networks"],
                "concern": ["supply chain issues", "high installation costs", "land use conflict",
                             "intermittency challenges", "investment needs"],
                "event":   ["a major blackout", "a green energy summit", "subsidy cuts",
                             "a new plant opening", "record energy demand"],
            },
            "Wildlife & Ecosystems": {
                "issue":   ["habitat destruction", "species decline", "deforestation rates",
                             "coral bleaching", "ocean acidification",
                             "biodiversity collapse", "wetland loss"],
                "target":  ["endangered species", "marine ecosystems", "forest cover",
                             "migratory birds", "reef systems"],
                "concern": ["illegal logging", "poaching pressure", "invasive species",
                             "land conversion", "fishing overcapacity"],
                "event":   ["a mass die-off", "a conservation report", "a protected area closure",
                             "a rewilding project", "a wildlife census"],
            },
            "Extreme Weather": {
                "issue":   ["flood risk", "heatwave intensity", "hurricane frequency",
                             "drought severity", "wildfire spread",
                             "storm surge", "unpredictable rainfall"],
                "target":  ["coastal cities", "agricultural regions", "mountain communities",
                             "island nations", "river basins"],
                "concern": ["infrastructure damage", "displacement", "crop failure",
                             "water scarcity", "health risks"],
                "event":   ["a category 5 hurricane", "a record heatwave", "major flooding",
                             "a prolonged drought", "destructive wildfires"],
            },
            "Pollution": {
                "issue":   ["plastic accumulation", "air quality deterioration", "toxic runoff",
                             "industrial waste discharge", "microplastic contamination",
                             "heavy metal pollution", "chemical spills"],
                "target":  ["water supplies", "urban air quality", "marine life",
                             "soil composition", "public health"],
                "concern": ["regulatory gaps", "corporate accountability", "health consequences",
                             "cleanup costs", "monitoring failures"],
                "event":   ["an industrial spill", "a smog alert", "a contamination scandal",
                             "a waste dumping incident", "a water quality crisis"],
            },
        }

        # Sujets neutres partagés entre toutes les catégories
        subjects = [
            "researchers", "local authorities", "environmental groups",
            "government officials", "international observers",
            "a new report", "advocacy organizations", "independent scientists",
        ]
        regions = [
            "Southeast Asia", "Western Europe", "sub-Saharan Africa",
            "the Pacific Islands", "Latin America", "the Arctic region",
            "South Asia", "North America", "the Mediterranean basin",
        ]

        rows = []
        for cat in CATEGORIES:
            cat_data = topics_by_cat[cat]
            for _ in range(300):
                t1 = rng.choice(templates)
                t2 = rng.choice(templates)

                def fill(t):
                    return t.format(
                        subject = rng.choice(subjects),
                        issue   = rng.choice(cat_data["issue"]),
                        target  = rng.choice(cat_data["target"]),
                        concern = rng.choice(cat_data["concern"]),
                        event   = rng.choice(cat_data["event"]),
                        region  = rng.choice(regions),
                    )

                title       = fill(t1)
                description = fill(t1).capitalize() + ". " + fill(t2).capitalize() + "."
                rows.append({"title": title, "description": description, "category": cat})

        df = pd.DataFrame(rows)

print(df.shape)
df.head()


## 4. Étiquetage thématique (classification) et préparation des résumés (Seq2Seq)

In [ ]:
def assign_category(text):
    text_l = text.lower()
    scores = {cat: sum(1 for kw in kws if kw in text_l) for cat, kws in KEYWORDS.items()}
    best_cat = max(scores, key=scores.get)
    if scores[best_cat] == 0:
        return None
    return best_cat

if "category" not in df.columns:
    df["category"] = (df["title"] + " " + df["description"]).apply(assign_category)
    df = df.dropna(subset=["category"]).reset_index(drop=True)

print(df["category"].value_counts())
print(df.shape)


In [ ]:
# Tâche Seq2Seq : (description complète) -> (résumé = titre)
df_seq2seq = df[["description", "title"]].rename(
    columns={"description": "source", "title": "target"}).reset_index(drop=True)
df_seq2seq.head()


## 5. Tokenisation et construction du vocabulaire

Tokens spéciaux : `<PAD>` (padding), `<UNK>` (mot inconnu), `<SOS>` (start of sequence),
`<EOS>` (end of sequence).


In [ ]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

class Vocabulary:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}

    def build(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(tokenize(t))

        specials = [PAD, UNK, SOS, EOS]
        vocab_words = specials + [w for w, c in counter.items() if c >= self.min_freq]
        self.word2idx = {w: i for i, w in enumerate(vocab_words)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        return self

    def __len__(self):
        return len(self.word2idx)

    def encode(self, text, add_sos_eos=False, max_len=None):
        tokens = tokenize(text)
        ids = [self.word2idx.get(t, self.word2idx[UNK]) for t in tokens]
        if add_sos_eos:
            ids = [self.word2idx[SOS]] + ids + [self.word2idx[EOS]]
        if max_len is not None:
            ids = ids[:max_len]
        return ids

    def decode(self, ids, skip_special=True):
        words = [self.idx2word.get(i, UNK) for i in ids]
        if skip_special:
            words = [w for w in words if w not in (PAD, SOS, EOS)]
        return " ".join(words)


# Vocabulaire commun pour la classification (titres)
class_vocab = Vocabulary(min_freq=1).build(df["title"].tolist())
print("Taille vocabulaire (classification) :", len(class_vocab))

# Vocabulaire pour le Seq2Seq (source + target)
seq2seq_vocab = Vocabulary(min_freq=1).build(
    df_seq2seq["source"].tolist() + df_seq2seq["target"].tolist())
print("Taille vocabulaire (seq2seq) :", len(seq2seq_vocab))


## 6. Préparation des données — Tâche de classification

In [ ]:
le_cat = {cat: i for i, cat in enumerate(CATEGORIES)}
MAX_LEN_CLS = 20

class TitleDataset(Dataset):
    def __init__(self, titles, categories, vocab, max_len=MAX_LEN_CLS):
        self.vocab = vocab
        self.max_len = max_len
        self.data = []
        for title, cat in zip(titles, categories):
            ids = vocab.encode(title, max_len=max_len)
            length = len(ids)
            ids = ids + [vocab.word2idx[PAD]] * (max_len - length)
            self.data.append((torch.tensor(ids, dtype=torch.long), length, le_cat[cat]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["category"],
                                       random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["category"],
                                    random_state=SEED)

train_cls_ds = TitleDataset(train_df["title"], train_df["category"], class_vocab)
val_cls_ds = TitleDataset(val_df["title"], val_df["category"], class_vocab)
test_cls_ds = TitleDataset(test_df["title"], test_df["category"], class_vocab)

BATCH_SIZE = 32
train_cls_loader = DataLoader(train_cls_ds, batch_size=BATCH_SIZE, shuffle=True)
val_cls_loader = DataLoader(val_cls_ds, batch_size=BATCH_SIZE, shuffle=False)
test_cls_loader = DataLoader(test_cls_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_cls_ds)} | Val: {len(val_cls_ds)} | Test: {len(test_cls_ds)}")


## 7. Implémentation des architectures récurrentes : RNN, LSTM, GRU

Les trois architectures partagent la même structure générale (embedding → couche récurrente →
classification sur le dernier état caché), seule la cellule récurrente change.


In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes, cell_type="LSTM",
                 num_layers=1, pad_idx=0, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)

        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[cell_type]
        self.cell_type = cell_type
        self.rnn = rnn_cls(embed_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))  # (B, T, E)
        if self.cell_type == "LSTM":
            _, (h_n, _) = self.rnn(emb)
        else:
            _, h_n = self.rnn(emb)
        last_hidden = self.dropout(h_n[-1])  # (B, H)
        return self.fc(last_hidden)


## 8. Entraînement comparatif : RNN vs LSTM vs GRU

Métriques suivies par epoch : loss, accuracy, **perplexité** (sur la tâche de classification,
calculée comme $\exp(\text{loss})$ à titre d'illustration), et **norme moyenne du gradient**
(pour analyser le BPTT et l'effet du gradient clipping).


In [ ]:
def train_recurrent(model, train_loader, val_loader, n_epochs=10, lr=1e-3,
                     clip_value=None):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    history = {"val_acc": [], "perplexity": [], "grad_norm": []}

    for epoch in range(n_epochs):
        model.train()
        running_loss, grad_norms = 0.0, []
        for xb, lengths, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()

            total_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            grad_norms.append(total_norm ** 0.5)

            if clip_value is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip_value)

            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for xb, lengths, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                loss = criterion(out, yb)
                val_loss += loss.item() * xb.size(0)
                correct += (out.argmax(1) == yb).sum().item()
        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)

        history["val_acc"].append(val_acc)
        history["perplexity"].append(math.exp(val_loss))
        history["grad_norm"].append(np.mean(grad_norms))

    return model, history


In [ ]:
VOCAB_SIZE_CLS = len(class_vocab)
EMBED_DIM, HIDDEN_DIM, N_CLASSES = 64, 128, len(CATEGORIES)

results_rnn = {}
for cell_type in ["RNN", "LSTM", "GRU"]:
    torch.manual_seed(SEED)
    model = RecurrentClassifier(VOCAB_SIZE_CLS, EMBED_DIM, HIDDEN_DIM, N_CLASSES,
                                 cell_type=cell_type, dropout=0.4,
                                 pad_idx=class_vocab.word2idx[PAD])
    trained_model, history = train_recurrent(model, train_cls_loader, val_cls_loader,
                                              n_epochs=10)
    results_rnn[cell_type] = (trained_model, history)
    print(f"{cell_type:5s} -> val_acc finale = {history['val_acc'][-1]:.4f} "
          f"| perplexité finale = {history['perplexity'][-1]:.3f}")


## 9. BPTT et effet du gradient clipping

La rétropropagation à travers le temps (**BPTT**) déroule la récurrence sur $T$ pas de temps,
ce qui peut provoquer une **explosion ou un vanishing des gradients**. On compare
l'entraînement d'un RNN simple sans clipping, avec clip=1.0 et clip=5.0.


In [ ]:
clip_results = {}
for clip_val, label in [(None, "Sans clipping"), (1.0, "Clip = 1.0"), (5.0, "Clip = 5.0")]:
    torch.manual_seed(SEED)
    model = RecurrentClassifier(VOCAB_SIZE_CLS, EMBED_DIM, HIDDEN_DIM, N_CLASSES,
                                 cell_type="RNN", dropout=0.4,
                                 pad_idx=class_vocab.word2idx[PAD])
    trained_model, history = train_recurrent(model, train_cls_loader, val_cls_loader,
                                              n_epochs=10, clip_value=clip_val)
    clip_results[label] = history
    print(f"{label:15s} -> grad_norm moyen = {np.mean(history['grad_norm']):.3f} "
          f"| val_acc finale = {history['val_acc'][-1]:.4f}")

# ── Plot : norme du gradient uniquement (comme le collègue) ──
clip_colors = ["#e74c3c", "#3498db", "#2ecc71"]
plt.figure(figsize=(8, 4.5))
for (label, h), color in zip(clip_results.items(), clip_colors):
    plt.plot(h["grad_norm"], color=color, lw=1.5, alpha=0.8, label=label)
plt.xlabel("Époque"); plt.ylabel("||grad||")
plt.title("Norme du Gradient — Effet du Clipping", fontweight="bold")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("gradient_clipping.png", dpi=120)
plt.show()


**Observation attendue** : sans clipping, la norme du gradient peut présenter des pics
importants (instabilité du BPTT sur des séquences de longueur 20). Le clipping à 1.0 stabilise
fortement la norme du gradient au prix d'une légère réduction de la vitesse d'apprentissage ;
clip=5.0 offre un compromis intermédiaire. La stabilité accrue se traduit généralement par une
convergence plus régulière de la validation accuracy.


## 11. Préparation des données — Tâche Seq2Seq (résumé automatique)

Chaque description (séquence source) est associée à un titre (séquence cible / résumé).
Padding et masquage sont gérés via `<PAD>`, encadrement par `<SOS>`/`<EOS>` pour la cible.


In [ ]:
MAX_LEN_SRC = 30
MAX_LEN_TGT = 16
PAD_IDX = seq2seq_vocab.word2idx[PAD]

class Seq2SeqDataset(Dataset):
    def __init__(self, sources, targets, vocab, max_len_src=MAX_LEN_SRC, max_len_tgt=MAX_LEN_TGT):
        self.data = []
        for src, tgt in zip(sources, targets):
            src_ids = vocab.encode(src, max_len=max_len_src)
            src_ids += [vocab.word2idx[PAD]] * (max_len_src - len(src_ids))

            tgt_ids = vocab.encode(tgt, add_sos_eos=True, max_len=max_len_tgt)
            tgt_ids += [vocab.word2idx[PAD]] * (max_len_tgt - len(tgt_ids))

            self.data.append((torch.tensor(src_ids, dtype=torch.long),
                               torch.tensor(tgt_ids, dtype=torch.long)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_s2s, temp_s2s = train_test_split(df_seq2seq, test_size=0.30, random_state=SEED)
val_s2s, test_s2s = train_test_split(temp_s2s, test_size=0.50, random_state=SEED)

train_s2s_ds = Seq2SeqDataset(train_s2s["source"], train_s2s["target"], seq2seq_vocab)
val_s2s_ds = Seq2SeqDataset(val_s2s["source"], val_s2s["target"], seq2seq_vocab)
test_s2s_ds = Seq2SeqDataset(test_s2s["source"], test_s2s["target"], seq2seq_vocab)

train_s2s_loader = DataLoader(train_s2s_ds, batch_size=32, shuffle=True)
val_s2s_loader = DataLoader(val_s2s_ds, batch_size=32, shuffle=False)
test_s2s_loader = DataLoader(test_s2s_ds, batch_size=32, shuffle=False)

print(f"Train: {len(train_s2s_ds)} | Val: {len(val_s2s_ds)} | Test: {len(test_s2s_ds)}")


## 12. Architecture Seq2Seq : encodeur-décodeur LSTM avec teacher forcing


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))
        outputs, (h, c) = self.lstm(emb)
        return h, c


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_token, h, c):
        emb = self.dropout(self.embedding(input_token))  # (B, 1, E)
        output, (h, c) = self.lstm(emb, (h, c))
        logits = self.fc(output.squeeze(1))  # (B, V)
        return logits, h, c


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, vocab, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.vocab = vocab
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)
        h, c = self.encoder(src)

        input_token = tgt[:, 0].unsqueeze(1)  # <SOS>
        for t in range(1, tgt_len):
            logits, h, c = self.decoder(input_token, h, c)
            outputs[:, t] = logits
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = logits.argmax(1).unsqueeze(1)
            input_token = tgt[:, t].unsqueeze(1) if teacher_force else top1

        return outputs


## 13. Entraînement du modèle Seq2Seq

In [ ]:
EMBED_DIM_S2S, HIDDEN_DIM_S2S = 128, 256
VOCAB_SIZE_S2S = len(seq2seq_vocab)
SOS_IDX = seq2seq_vocab.word2idx[SOS]
EOS_IDX = seq2seq_vocab.word2idx[EOS]

torch.manual_seed(SEED)
encoder = Encoder(VOCAB_SIZE_S2S, EMBED_DIM_S2S, HIDDEN_DIM_S2S, PAD_IDX, dropout=0.3)
decoder = Decoder(VOCAB_SIZE_S2S, EMBED_DIM_S2S, HIDDEN_DIM_S2S, PAD_IDX, dropout=0.3)
seq2seq_model = Seq2Seq(encoder, decoder, seq2seq_vocab, DEVICE).to(DEVICE)

criterion_s2s = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer_s2s = optim.Adam(seq2seq_model.parameters(), lr=1e-3, weight_decay=1e-5)

N_EPOCHS_S2S = 20
s2s_losses = []

for epoch in range(1, N_EPOCHS_S2S + 1):
    seq2seq_model.train()
    running_loss = 0.0
    for src, tgt in train_s2s_loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        optimizer_s2s.zero_grad()
        outputs = seq2seq_model(src, tgt, teacher_forcing_ratio=0.5)
        loss = criterion_s2s(outputs[:, 1:].reshape(-1, VOCAB_SIZE_S2S),
                              tgt[:, 1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(seq2seq_model.parameters(), 1.0)
        optimizer_s2s.step()
        running_loss += loss.item() * src.size(0)

    train_loss = running_loss / len(train_s2s_loader.dataset)
    s2s_losses.append(train_loss)

    if epoch % 5 == 0:
        ppl = math.exp(min(train_loss, 20))
        print(f"Époque {epoch:>3}/{N_EPOCHS_S2S} | Loss: {train_loss:.4f} | Perplexité: {ppl:.2f}")

# ── Plot : train loss + perplexité (comme le collègue) ──
perplexities = [math.exp(min(l, 20)) for l in s2s_losses]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Entraînement Seq2Seq — Encodeur-Décodeur LSTM", fontsize=13, fontweight="bold")

axes[0].plot(s2s_losses, "b-", lw=2, label="Train Loss")
axes[0].set_title("Loss (entraînement)", fontweight="bold")
axes[0].set_xlabel("Époque"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(perplexities, "r-", lw=2)
axes[1].fill_between(range(len(perplexities)), perplexities, alpha=0.1, color="red")
axes[1].set_title("Perplexité (plus basse = meilleur)", fontweight="bold")
axes[1].set_xlabel("Époque"); axes[1].set_ylabel("Perplexité"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("seq2seq_loss.png", dpi=120)
plt.show()
print("Entraînement Seq2Seq terminé.")


## 14. Stratégies de décodage : glouton et beam search


In [ ]:
def greedy_decode(model, src, vocab, max_len=MAX_LEN_TGT):
    model.eval()
    with torch.no_grad():
        h, c = model.encoder(src.unsqueeze(0).to(model.device))
        input_token = torch.tensor([[SOS_IDX]]).to(model.device)
        result = []
        for _ in range(max_len):
            logits, h, c = model.decoder(input_token, h, c)
            top1 = logits.argmax(1).item()
            if top1 == EOS_IDX:
                break
            result.append(top1)
            input_token = torch.tensor([[top1]]).to(model.device)
    return vocab.decode(result)


def beam_search_decode(model, src, vocab, beam_width=3, max_len=MAX_LEN_TGT):
    model.eval()
    with torch.no_grad():
        h0, c0 = model.encoder(src.unsqueeze(0).to(model.device))
        # Chaque candidat : (séquence de tokens, score log-proba cumulé, h, c, terminé?)
        beams = [([SOS_IDX], 0.0, h0, c0, False)]

        for _ in range(max_len):
            new_beams = []
            for seq, score, h, c, done in beams:
                if done:
                    new_beams.append((seq, score, h, c, done))
                    continue
                input_token = torch.tensor([[seq[-1]]]).to(model.device)
                logits, h_new, c_new = model.decoder(input_token, h, c)
                log_probs = F.log_softmax(logits, dim=1).squeeze(0)
                topk_probs, topk_idx = log_probs.topk(beam_width)
                for k in range(beam_width):
                    token = topk_idx[k].item()
                    new_score = score + topk_probs[k].item()
                    new_seq = seq + [token]
                    new_done = (token == EOS_IDX)
                    new_beams.append((new_seq, new_score, h_new, c_new, new_done))

            # Garder les meilleurs `beam_width` candidats (normalisation par longueur)
            new_beams.sort(key=lambda x: x[1] / len(x[0]), reverse=True)
            beams = new_beams[:beam_width]

            if all(b[4] for b in beams):
                break

        best_seq = beams[0][0]
    return vocab.decode(best_seq)


In [ ]:
# Exemples qualitatifs de résumés générés
seq2seq_model.eval()
print("=== Exemples de résumés générés (décodage glouton vs beam search) ===\n")
for i in range(5):
    src, tgt = test_s2s_ds[i]
    src_text = seq2seq_vocab.decode(src.tolist())
    tgt_text = seq2seq_vocab.decode(tgt.tolist())
    greedy_out = greedy_decode(seq2seq_model, src, seq2seq_vocab)
    beam_out = beam_search_decode(seq2seq_model, src, seq2seq_vocab, beam_width=3)

    print(f"Source   : {src_text}")
    print(f"Référence: {tgt_text}")
    print(f"Glouton  : {greedy_out}")
    print(f"Beam(3)  : {beam_out}")
    print("-"*80)


## 15. Évaluation BLEU : décodage glouton vs beam search

In [ ]:
def evaluate_bleu(model, dataset, vocab, decode_fn, n_samples=200, **decode_kwargs):
    scores = []
    n = min(n_samples, len(dataset))
    for i in range(n):
        src, tgt = dataset[i]
        ref = vocab.decode(tgt.tolist()).split()
        hyp = decode_fn(model, src, vocab, **decode_kwargs).split()
        if len(hyp) == 0:
            hyp = [UNK]
        score = sentence_bleu([ref], hyp, weights=(1.0, 0, 0, 0), smoothing_function=smoothie)
        scores.append(score)
    return np.mean(scores)

bleu_greedy = evaluate_bleu(seq2seq_model, test_s2s_ds, seq2seq_vocab, greedy_decode)
bleu_beam2 = evaluate_bleu(seq2seq_model, test_s2s_ds, seq2seq_vocab, beam_search_decode, beam_width=2)
bleu_beam3 = evaluate_bleu(seq2seq_model, test_s2s_ds, seq2seq_vocab, beam_search_decode, beam_width=3)
bleu_beam5 = evaluate_bleu(seq2seq_model, test_s2s_ds, seq2seq_vocab, beam_search_decode, beam_width=5)

print(f"BLEU-1 (glouton)   : {bleu_greedy:.4f}")
print(f"BLEU-1 (beam k=2)  : {bleu_beam2:.4f}")
print(f"BLEU-1 (beam k=3)  : {bleu_beam3:.4f}")
print(f"BLEU-1 (beam k=5)  : {bleu_beam5:.4f}")

improvement = (bleu_beam3 - bleu_greedy) / bleu_greedy * 100
print(f"\nAmélioration relative beam(3) vs glouton : {improvement:+.1f}%")


In [ ]:
plt.figure(figsize=(7,4.5))
strategies = ["Greedy", "Beam k=2", "Beam k=3", "Beam k=5"]
scores = [bleu_greedy, bleu_beam2, bleu_beam3, bleu_beam5]
plt.bar(strategies, scores, color="seagreen")
plt.ylabel("BLEU-1 score")
plt.title("Comparaison des stratégies de décodage - Seq2Seq")
plt.savefig("bleu_comparison.png", dpi=120)
plt.show()
